# USAspending Prototype Report Notebook

This notebook is the handoff report for the refined USAspending data pull process. It is intended for an analyst who needs to understand what the script does, how to run it safely, what files it creates, and how those files support a Power BI prototype.

The notebook does not replace the script. The script, `usaspending_data_pull_refined.py`, is the source of truth for the data pull. This notebook explains and demonstrates the workflow around that script.

At a high level, the workflow is:

1. Start with an Excel list of company names.
2. Search USAspending for likely parent company matches.
3. Search for related child or subsidiary entities.
4. Combine the parent and child records into a hierarchy.
5. Pull award records for the matched UEIs by date window.
6. Create normalized model tables and readable Power BI exports.
7. Review summary counts, failed requests, and prototype visuals.

Current test input: `CompanyNames.xlsx`.

Important handoff note: the notebook is designed so an analyst can review existing output files without accidentally rerunning API calls. API calls only run from this notebook when `RUN_PIPELINE = True` in Step 3.


## 1. Setup

This step prepares the notebook environment and defines the file paths used throughout the report.

What this step does:

- Imports the Python packages used by the notebook, mainly `pandas` and standard-library utilities.
- Sets `REPO_ROOT` to the current working folder.
- Defines the expected input file, script file, config file, and output CSV paths.
- Creates the Power BI prototype output folder if it does not already exist.
- Prints the notebook start time and repository root so the analyst can confirm they are running from the correct folder.

What this step does not do:

- It does not call the USAspending API.
- It does not modify the main data files.
- It does not run the refined script.

Why it matters: most downstream cells read from the paths defined here. If the notebook is opened from the wrong folder, later cells may show missing files even though the files exist somewhere else.


In [31]:
from __future__ import annotations

import subprocess
import sys
from datetime import datetime
from html import escape
from pathlib import Path

import pandas as pd

try:
    from IPython.display import HTML, Markdown, display
except Exception:
    HTML = Markdown = None
    display = print

REPO_ROOT = Path.cwd()
COMPANY_INPUT = REPO_ROOT / "CompanyNames.xlsx"
SCRIPT_PATH = REPO_ROOT / "usaspending_data_pull_refined.py"
CONFIG_PATH = REPO_ROOT / "config.json"

OUTPUTS = {
    "parents": REPO_ROOT / "parent_companies_ueis_duns.csv",
    "children": REPO_ROOT / "child_companies_duns_ueis.csv",
    "hierarchy": REPO_ROOT / "company_hierarchy.csv",
    "entities": REPO_ROOT / "entity_master.csv",
    "relationships": REPO_ROOT / "relationships.csv",
    "awards": REPO_ROOT / "award_fact.csv",
    "award_progress": REPO_ROOT / "award_progress.csv",
    "failed_awards": REPO_ROOT / "failed_award_requests.csv",
    "request_log": REPO_ROOT / "award_request_log.csv",
    "run_log": REPO_ROOT / "run_log.csv",
}

POWERBI_DIR = REPO_ROOT / "output" / "powerbi_prototype"
POWERBI_DIR.mkdir(parents=True, exist_ok=True)

print(f"Notebook run started: {datetime.now().isoformat(timespec='seconds')}")
print(f"Repository root: {REPO_ROOT}")

Notebook run started: 2026-05-05T15:07:14
Repository root: c:\Users\tonya\Repos\at-risk-materials


## 2. Confirm Test Companies

This step opens the Excel input file and shows the company names that will be used by the pipeline.

Expected input:

- File: `CompanyNames.xlsx`
- Preferred column: `Company`
- Fallback behavior: if there is no `Company` column, the notebook uses the first column in the workbook.

Why this check is important:

- The company names drive the parent-match search.
- Broad names can return unrelated USAspending entities, so the analyst should review the input before running the API steps.
- Blank values are ignored, but misspellings or inconsistent legal names can affect match quality.

If the displayed company list is wrong, stop here and correct `CompanyNames.xlsx` before running Step 3.


In [32]:
if not COMPANY_INPUT.exists():
    raise FileNotFoundError(f"Missing input file: {COMPANY_INPUT}")

company_df = pd.read_excel(COMPANY_INPUT)
display(company_df)

company_col = "Company" if "Company" in company_df.columns else company_df.columns[0]
companies = company_df[company_col].dropna().astype(str).str.strip().tolist()
print(f"Companies to test: {len(companies)}")
print(companies)

,Company
0,3M Company
1,Dow INC


Companies to test: 2
['3M Company', 'Dow INC']


## 3. Run the Refined Pipeline

This step is the controlled launch point for `usaspending_data_pull_refined.py` from inside the notebook.

The key switch is `RUN_PIPELINE`:

- `RUN_PIPELINE = False` means the notebook only displays the commands that would run. No API calls are made.
- `RUN_PIPELINE = True` means the notebook runs the selected pipeline commands.

The key run mode is `PIPELINE_RUN_MODE`:

| Mode | Script steps run | When to use it |
|---|---|---|
| `full` | `parents`, `children`, `combine`, `awards`, `powerbi-exports` | Use for a complete refresh from the Excel input through Power BI-ready files. |
| `setup_only` | `parents`, `children`, `combine` | Use when you only want to build or inspect the company hierarchy before pulling awards. |
| `awards_only` | `awards` | Use after an awards API timeout or partial failure. This lets the script resume incomplete award windows without rerunning the hierarchy setup. |
| `custom` | Whatever is listed in `CUSTOM_PIPELINE_STEPS` | Use for targeted troubleshooting or one-off refreshes. |

What each script step does:

| Step | Purpose | Main outputs |
|---|---|---|
| `parents` | Searches USAspending for parent company matches from `CompanyNames.xlsx`. | `parent_companies_ueis_duns.csv` |
| `children` | Looks for child/subsidiary entities associated with the matched parents. | `child_companies_duns_ueis.csv` |
| `combine` | Builds the combined hierarchy and normalized model tables. | `company_hierarchy.csv`, `entity_master.csv`, `relationships.csv` |
| `awards` | Pulls award records by UEI and date window, with progress and failure logging. | `award_fact.csv`, `award_progress.csv`, `award_request_log.csv`, `failed_award_requests.csv` |
| `powerbi-exports` | Creates analyst-friendly readable exports for dashboard building. | `output/powerbi_prototype/*.csv` |

Partial awards failures: if the awards step reports `status=partial_failure`, successful windows are still saved. The recovery path is normally to rerun Step 3 with `PIPELINE_RUN_MODE = "awards_only"`, then run or include `powerbi-exports` after awards complete.

For a full command-line run outside the notebook, use:

```bash
python usaspending_data_pull_refined.py --step all --input CompanyNames.xlsx --non-interactive
```


In [ ]:
RUN_PIPELINE = False

# Options: "full", "setup_only", "awards_only", or "custom".
# Use "awards_only" after an awards partial failure so resume can retry only incomplete windows.
PIPELINE_RUN_MODE = "full"
CUSTOM_PIPELINE_STEPS = ["awards"]
ALLOW_AWARDS_PARTIAL_FAILURE = True

pipeline_steps_by_mode = {
    "full": ["parents", "children", "combine", "awards", "powerbi-exports"],
    "setup_only": ["parents", "children", "combine"],
    "awards_only": ["awards"],
    "custom": CUSTOM_PIPELINE_STEPS,
}

if PIPELINE_RUN_MODE not in pipeline_steps_by_mode:
    raise ValueError(f"Unsupported PIPELINE_RUN_MODE: {PIPELINE_RUN_MODE}")

pipeline_steps = pipeline_steps_by_mode[PIPELINE_RUN_MODE]

commands = []
for step in pipeline_steps:
    command = [sys.executable, "-B", str(SCRIPT_PATH), "--step", step, "--non-interactive"]
    if step == "parents":
        command.extend(["--input", str(COMPANY_INPUT)])
    commands.append(command)

print(f"PIPELINE_RUN_MODE = {PIPELINE_RUN_MODE}")
for command in commands:
    print(" ".join(command))

if RUN_PIPELINE:
    for command in commands:
        print(f"\nRunning: {' '.join(command)}")
        completed = subprocess.run(command, cwd=REPO_ROOT, text=True, capture_output=True)
        print(completed.stdout)
        if completed.stderr:
            print(completed.stderr)
        command_text = " ".join(command)
        combined_output = f"{completed.stdout}\n{completed.stderr}"
        awards_partial_failure = (
            completed.returncode != 0
            and "--step awards" in command_text
            and "status=partial_failure" in combined_output
        )
        if awards_partial_failure and ALLOW_AWARDS_PARTIAL_FAILURE:
            print("\nAwards had a partial API failure. Existing successful windows were saved.")
            print("Rerun this cell with PIPELINE_RUN_MODE = 'awards_only' to retry only incomplete award windows.")
            break
        completed.check_returncode()
else:
    print("\nRUN_PIPELINE is False, so no API calls were made from this notebook.")


PIPELINE_RUN_MODE = awards_only
c:\Users\tonya\Repos\at-risk-materials\.venv\Scripts\python.exe -B c:\Users\tonya\Repos\at-risk-materials\usaspending_data_pull_refined.py --step awards --non-interactive

RUN_PIPELINE is False, so no API calls were made from this notebook.


## 4. What the Output Files Mean

This step reads the output CSV files if they exist and summarizes their row counts, column counts, and last modified times.

The pipeline creates three categories of files:

| Category | Files | Meaning |
|---|---|---|
| Discovery/staging files | `parent_companies_ueis_duns.csv`, `child_companies_duns_ueis.csv`, `company_hierarchy.csv` | These explain how the script moved from input company names to USAspending entities. They are useful for audit and troubleshooting. |
| Normalized model files | `entity_master.csv`, `relationships.csv`, `award_fact.csv` | These are the governed core files for analysis and Power BI modeling. |
| Operational log files | `award_progress.csv`, `award_request_log.csv`, `failed_award_requests.csv`, `run_log.csv` | These show what was attempted, what succeeded, what failed, and whether a rerun is needed. |

Plain-language file definitions:

- `parent_companies_ueis_duns.csv`: matched parent companies and their USAspending identifiers.
- `child_companies_duns_ueis.csv`: child or subsidiary entities found from the parent matches.
- `company_hierarchy.csv`: combined parent and child list used as the input to the awards pull.
- `entity_master.csv`: one row per unique entity/UEI, including names and parent fields.
- `relationships.csv`: child-to-parent links using UEIs.
- `award_fact.csv`: award-level facts such as award ID, recipient UEI, parent UEI, amount, agencies, and dates.
- `award_progress.csv`: one row per attempted UEI/date window so the awards step can resume.
- `award_request_log.csv`: detailed request log for awards API troubleshooting.
- `failed_award_requests.csv`: requests that failed after retries or were left pending.
- `run_log.csv`: high-level run history for each script step.

If a file is missing in this table, it usually means the corresponding script step has not been run yet or failed before producing output.


In [34]:
def read_csv_if_exists(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(path, dtype=str)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

data = {name: read_csv_if_exists(path) for name, path in OUTPUTS.items()}

output_status = pd.DataFrame(
    [
        {
            "output": name,
            "file": path.name,
            "exists": path.exists(),
            "rows": len(data[name]),
            "columns": len(data[name].columns),
            "last_modified": datetime.fromtimestamp(path.stat().st_mtime).isoformat(timespec="seconds") if path.exists() else "",
        }
        for name, path in OUTPUTS.items()
    ]
)

display(output_status)

,output,file,exists,rows,columns,last_modified
0,parents,parent_companies_ueis_duns.csv,True,2,14,2026-05-05T14:49:33
1,children,child_companies_duns_ueis.csv,True,44,8,2026-05-05T14:49:38
2,hierarchy,company_hierarchy.csv,True,46,5,2026-05-05T14:49:39
3,entities,entity_master.csv,True,45,7,2026-05-05T14:49:39
4,relationships,relationships.csv,True,43,6,2026-05-05T14:49:39
5,awards,award_fact.csv,True,5384,8,2026-05-05T12:37:21
6,award_progress,award_progress.csv,True,270,8,2026-05-05T12:37:22
7,failed_awards,failed_award_requests.csv,False,0,0,
8,request_log,award_request_log.csv,True,392,13,2026-05-05T12:37:22
9,run_log,run_log.csv,True,6,15,2026-05-05T14:49:41


## 5. Prototype Summary Metrics

This step turns the output files into a short metrics table for reporting status.

The metrics are intentionally simple:

- `Companies in input`: number of company names loaded from `CompanyNames.xlsx`.
- `Parent rows found`: number of matched parent rows created by the `parents` step.
- `Child rows found`: number of child/subsidiary rows created by the `children` step.
- `Hierarchy rows`: total rows available to the hierarchy and awards process.
- `Unique entities`: distinct UEIs in `entity_master.csv`.
- `Relationship rows`: child-parent links in `relationships.csv`.
- `Award rows`: award records written to `award_fact.csv`.
- `Award amount total`: sum of award amounts currently available in `award_fact.csv`.
- `Failed award request rows`: count of requests in `failed_award_requests.csv`.

How to interpret this step:

- Zero rows can be normal before the pipeline has been run.
- A nonzero failed request count does not always mean all data is bad; it may mean only some UEI/date windows need to be retried.
- If award rows look lower than expected, check `run_log.csv`, `award_progress.csv`, and `failed_award_requests.csv` before rerunning the full pipeline.


In [35]:
def nunique_clean(df: pd.DataFrame, column: str) -> int:
    if df.empty or column not in df.columns:
        return 0
    return df[column].dropna().astype(str).str.strip().replace("", pd.NA).dropna().nunique()

awards = data["awards"]
award_amount = pd.to_numeric(awards.get("award_amount", pd.Series(dtype=str)), errors="coerce").sum() if not awards.empty else 0

summary = pd.DataFrame(
    [
        {"metric": "Companies in input", "value": len(companies)},
        {"metric": "Parent rows found", "value": len(data["parents"])},
        {"metric": "Child rows found", "value": len(data["children"])},
        {"metric": "Hierarchy rows", "value": len(data["hierarchy"])},
        {"metric": "Unique entities", "value": nunique_clean(data["entities"], "uei")},
        {"metric": "Relationship rows", "value": len(data["relationships"])},
        {"metric": "Award rows", "value": len(awards)},
        {"metric": "Award amount total", "value": round(float(award_amount), 2)},
        {"metric": "Failed award request rows", "value": len(data["failed_awards"])},
    ]
)

display(summary)

,metric,value
0,Companies in input,2.000000e+00
1,Parent rows found,2.000000e+00
2,Child rows found,4.400000e+01
3,Hierarchy rows,4.600000e+01
4,Unique entities,4.500000e+01
5,Relationship rows,4.300000e+01
6,Award rows,5.384000e+03
7,Award amount total,8.529209e+08
8,Failed award request rows,0.000000e+00


## 6. Workflow Graphic

This step gives a simple process view of the pipeline without requiring extra visualization libraries.

The first output is a pandas table with each workflow stage, its main output file, and its purpose. The second output is a plain-text lineage view that shows the order of the files from input to dashboard prototype.

Why this format is used:

- It works anywhere pandas is available.
- It is easy to copy into notes or status updates.
- It avoids dependency issues when Graphviz, Mermaid, Plotly, or other visualization packages are not available.

The workflow is intentionally linear here. In the actual data model, `entity_master.csv`, `relationships.csv`, and `award_fact.csv` connect through UEI keys rather than forming one long flat table.


In [ ]:
workflow = pd.DataFrame(
    [
        {"Step": 1, "Stage": "Input", "Output": "CompanyNames.xlsx", "Purpose": "Company list"},
        {"Step": 2, "Stage": "Parents", "Output": "parent_companies_ueis_duns.csv", "Purpose": "Official parent matches and UEIs"},
        {"Step": 3, "Stage": "Children", "Output": "child_companies_duns_ueis.csv", "Purpose": "Child/subsidiary entities"},
        {"Step": 4, "Stage": "Hierarchy", "Output": "company_hierarchy.csv", "Purpose": "Combined parent and child hierarchy"},
        {"Step": 5, "Stage": "Normalized tables", "Output": "entity_master.csv + relationships.csv + award_fact.csv", "Purpose": "Power BI model tables"},
        {"Step": 6, "Stage": "Reporting", "Output": "Power BI prototype", "Purpose": "Dashboard-ready output"},
    ]
)

display(workflow)

lineage_text = """
CompanyNames.xlsx
       |
parent_companies_ueis_duns.csv
       |
child_companies_duns_ueis.csv
       |
company_hierarchy.csv
       |
entity_master.csv + relationships.csv + award_fact.csv
       |
Power BI prototype
""".strip()

print(lineage_text)


### Optional Graphviz Workflow Preview

This optional cell renders the same workflow as a directed graph.

Dependency requirements:

- Python package: `graphviz`
- System executable: Graphviz `dot` available on `PATH`

If the Python package or the `dot` executable is missing, the cell prints setup guidance instead of stopping the notebook. This view is optional; the pandas workflow table above is the dependency-light version that should work for most analysts.

Use this cell when you want a cleaner visual for presentations or stakeholder review. Use the pandas table when the goal is reliability and portability.


In [ ]:
import shutil

try:
    import graphviz
except ImportError:
    print("Python package missing. Install it with: python -m pip install graphviz")
else:
    if shutil.which("dot") is None:
        print("Graphviz Python package is installed, but the Graphviz dot executable was not found on PATH.")
        print("Install Graphviz Desktop from https://graphviz.org/download/ and reopen VS Code, or add Graphviz\\bin to PATH.")
    else:
        graph = graphviz.Digraph("usaspending_workflow", format="svg")
        graph.attr(rankdir="TB", bgcolor="white", pad="0.35", nodesep="0.55", ranksep="0.65")
        graph.attr("node", shape="box", style="rounded,filled", color="#334155", fontname="Segoe UI", fontsize="11", margin="0.12,0.08")
        graph.attr("edge", color="#475569", arrowsize="0.8")

        graph.node("input", "CompanyNames.xlsx", fillcolor="#e0f2fe")
        graph.node("parents", "parent_companies_ueis_duns.csv", fillcolor="#f8fafc")
        graph.node("children", "child_companies_duns_ueis.csv", fillcolor="#f8fafc")
        graph.node("hierarchy", "company_hierarchy.csv", fillcolor="#f8fafc")
        graph.node("model", "entity_master.csv + relationships.csv + award_fact.csv", fillcolor="#ecfdf5")
        graph.node("powerbi", "Power BI prototype", fillcolor="#fef9c3")

        graph.edges([
            ("input", "parents"),
            ("parents", "children"),
            ("children", "hierarchy"),
            ("hierarchy", "model"),
            ("model", "powerbi"),
        ])

        display(graph)


## 7. How the Tables Are Normalized

Normalized means the pipeline separates different business concepts into separate tables instead of repeating everything in one large CSV.

In this project, the major concepts are:

- Entities: companies, parents, subsidiaries, and their identifiers.
- Relationships: how child entities connect to parent entities.
- Awards: federal award records tied to recipient UEIs and parent UEIs.

The normalized files are:

| Table | Grain | Key Columns | Purpose |
|---|---|---|---|
| `entity_master.csv` | One row per unique UEI/entity | `uei`, `ultimate_parent_uei` | Company/entity lookup table. Use it to get names and identifiers. |
| `relationships.csv` | One row per child-parent relationship | `child_uei`, `parent_uei` | Bridge table that links subsidiaries to parents. |
| `award_fact.csv` | One row per award record | `award_id`, `recipient_uei`, `ultimate_parent_uei` | Award-level fact table for amounts, dates, agencies, and recipients. |
| `company_hierarchy.csv` | Parent and child rows from discovery | `Original Company Name`, `UEI`, `Recipient Level` | Traceable hierarchy used before awards are pulled. |

Why this is better than one wide file:

- Company details are stored once in `entity_master.csv` instead of repeated on every award row.
- Child-parent logic is auditable in `relationships.csv`.
- Award analysis stays focused on award-level fields in `award_fact.csv`.
- Power BI can model the files with clear relationships instead of relying on duplicated text columns.

Recommended Power BI joins:

| From | To | Join Type |
|---|---|---|
| `award_fact.recipient_uei` | `entity_master.uei` | Many-to-one |
| `relationships.child_uei` | `entity_master.uei` | Many-to-one |
| `relationships.parent_uei` | `entity_master.uei` | Many-to-one |
| `award_fact.ultimate_parent_uei` | `entity_master.uei` | Many-to-one |

Important distinction: the normalized files are the governed model files. The readable Power BI exports in Step 9 are convenience files that add human-readable names for dashboard building.


## 8. Company Structure Graphic

This step creates a small parent-child structure graphic from the normalized relationship data.

Inputs used:

- `relationships.csv` for child-to-parent links.
- `entity_master.csv` for company names tied to UEIs.

What the graphic shows:

- One selected parent entity.
- A limited set of child entities connected to that parent.
- Display labels based on company names when available.

Limitations:

- This is a prototype visualization, not a complete entity-resolution audit.
- Very large company families may be trimmed so the graphic remains readable.
- If names are missing from `entity_master.csv`, the graphic may fall back to UEIs or shorter labels.

If this cell says the relationship data is missing, run the setup portion of the pipeline first: `parents`, `children`, and `combine`.


In [37]:
try:
    from IPython.display import HTML as _SVG_HTML, display as _svg_display
except Exception:
    _SVG_HTML = None
    _svg_display = None


def svg_box(label, x, y, w, h, fill="#f8fafc"):
    label = str(label).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    return (
        f'<g>'
        f'<rect x="{x}" y="{y}" width="{w}" height="{h}" rx="8" fill="{fill}" stroke="#cbd5e1" stroke-width="1" />'
        f'<text x="{x + 14}" y="{y + h / 2 + 5}" font-family="Segoe UI, Arial, sans-serif" font-size="13" fill="#0f172a">{label}</text>'
        f'</g>'
    )


def svg_arrow(x1, y1, x2, y2):
    return (
        f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="#475569" stroke-width="1.5" marker-end="url(#arrowhead)" />'
    )


def display_svg(svg: str):
    if _SVG_HTML is not None and _svg_display is not None:
        _svg_display(_SVG_HTML(svg))
    else:
        print(svg)

entities = data["entities"].copy()
relationships = data["relationships"].copy()

def entity_name_lookup(entities_df: pd.DataFrame) -> dict:
    if entities_df.empty or "uei" not in entities_df.columns:
        return {}
    name_col = "entity_name" if "entity_name" in entities_df.columns else None
    if not name_col:
        return {}
    lookup = {}
    for _, row in entities_df.iterrows():
        uei = str(row.get("uei", "")).strip()
        name = str(row.get(name_col, "")).strip()
        if uei:
            lookup[uei] = name or uei
    return lookup

name_by_uei = entity_name_lookup(entities)

def short_label(value: str, max_len: int = 34) -> str:
    value = str(value or "").strip()
    return value if len(value) <= max_len else value[: max_len - 1] + "..."

if relationships.empty or not {"child_uei", "parent_uei"}.issubset(relationships.columns):
    print("relationships.csv is not available yet. Run the hierarchy steps first.")
else:
    rel_preview = relationships[["child_uei", "parent_uei"]].dropna().drop_duplicates().head(14)
    parent_values = sorted({str(value).strip() for value in rel_preview["parent_uei"].dropna().tolist()})
    parent_uei = parent_values[0] if parent_values else "Parent"
    children = [str(value).strip() for value in rel_preview["child_uei"].dropna().tolist()]
    row_height = 48
    height = max(160, 70 + row_height * max(len(children), 1))
    parent_label = short_label(name_by_uei.get(parent_uei, parent_uei))
    child_boxes = []
    connectors = []
    for idx, child in enumerate(children):
        y = 35 + idx * row_height
        child_label = short_label(name_by_uei.get(child, child))
        child_boxes.append(svg_box(child_label, 430, y, 300, 36, fill="#f8fafc"))
        connectors.append(svg_arrow(350, height // 2, 430, y + 18))
    structure_svg = f'''
    <svg viewBox="0 0 780 {height}" width="100%" style="max-width: 780px; background: #ffffff; border: 1px solid #e5e7eb; border-radius: 8px;">
      <defs>
        <marker id="arrowhead" markerWidth="10" markerHeight="7" refX="9" refY="3.5" orient="auto">
          <polygon points="0 0, 10 3.5, 0 7" fill="#475569" />
        </marker>
      </defs>
      {svg_box(parent_label, 40, height // 2 - 29, 310, 58, fill='#ecfdf5')}
      {''.join(connectors)}
      {''.join(child_boxes)}
    </svg>
    '''
    display_svg(structure_svg)


## 9. Power BI Prototype Exports

This step displays the readable Power BI files created by the refined script.

The notebook no longer builds these readable files itself. That keeps the process consistent: the script creates the files, and the notebook reports whether they exist.

Expected folder:

- `output/powerbi_prototype`

Expected files:

| File | Description |
|---|---|
| `entity_master.csv` | Script-generated copy of the entity lookup table for the Power BI prototype folder. |
| `award_fact_readable.csv` | Award facts with readable company-name fields added, such as recipient and parent names. |
| `relationships_readable.csv` | Child-parent relationship rows with readable child and parent names added. |

To create or refresh these files manually, run:

```bash
python usaspending_data_pull_refined.py --step powerbi-exports --non-interactive
```

A full pipeline run also creates these files because `powerbi-exports` is the final step in `--step all`.


In [ ]:
powerbi_files = [
    POWERBI_DIR / "entity_master.csv",
    POWERBI_DIR / "award_fact_readable.csv",
    POWERBI_DIR / "relationships_readable.csv",
]

export_rows = []
for path in powerbi_files:
    if path.exists():
        try:
            df = pd.read_csv(path)
            export_rows.append({"file": path.name, "exists": True, "rows": len(df), "columns": len(df.columns), "folder": str(path.parent)})
        except pd.errors.EmptyDataError:
            export_rows.append({"file": path.name, "exists": True, "rows": 0, "columns": 0, "folder": str(path.parent)})
    else:
        export_rows.append({"file": path.name, "exists": False, "rows": 0, "columns": 0, "folder": str(path.parent)})

exports_df = pd.DataFrame(export_rows)
display(exports_df)
print(f"Power BI export folder: {POWERBI_DIR}")
print("To create or refresh these files, run: python usaspending_data_pull_refined.py --step powerbi-exports --non-interactive")


## 10. Power BI Prototype Plan

The first Power BI build should be an exploration dashboard, not a final production report. The goal is to prove that the data model supports useful questions and that the pipeline output is understandable to business users.

Recommended prototype pages:

1. **Overview**: total award amount, award count, company count, parent count, and failed request count.
2. **Company Explorer**: parent company, child entities, UEIs, award totals, and drill-through to award records.
3. **Agency View**: award totals and counts by awarding agency, funding agency, and sub-agency when available.
4. **Timeline View**: award activity by action date, start date, end date, or fiscal year depending on available columns.
5. **Data Quality View**: parent matches, ambiguous matches, failed award windows, and run-log status.

Recommended data approach:

- Use `entity_master.csv`, `relationships.csv`, and `award_fact.csv` as the governed model tables.
- Use `award_fact_readable.csv` and `relationships_readable.csv` when the prototype needs easier labels quickly.
- Keep `award_progress.csv`, `failed_award_requests.csv`, and `run_log.csv` available for refresh monitoring and troubleshooting.

The official dashboard build guide is `POWER_BI_READABLE_FILES_GUIDE.md`.


## 11. Why We Are Considering Bulk Downloads

The current refined script pulls awards by UEI and date window. This is traceable and easy to debug, but it can create many API calls when a company has many related entities or when the date range is split into many windows.

Why that matters:

- More API calls increase runtime.
- More API calls increase exposure to network resets, HTTP 500 responses, throttling, and timeouts.
- A partial API failure can leave some UEI/date windows incomplete even when most data was saved correctly.

The bulk download idea is different:

1. Download larger USAspending award files by date range.
2. Store a local copy and manifest.
3. Filter the downloaded award data to the UEIs in `entity_master.csv` or `company_hierarchy.csv`.
4. Produce an award fact file with fewer live API calls.

Why this may be better for dashboard refreshes:

- It shifts work from repeated API searches to local file processing.
- It can make refreshes more repeatable.
- It gives a clearer audit trail for which bulk files were downloaded and processed.

## 12. SEC Notebook Overview

The SEC notebook is a separate exploration track. It is related to this project because it may provide public-company financial context, but it does not use the same identifiers as USAspending.

Current SEC notebook:

- `sec_data_exploration_starter.ipynb`

What the SEC notebook explores:

- Ticker symbols and CIKs for public-company identification.
- SEC submissions metadata.
- XBRL company facts.
- Long-form financial facts and yearly company metrics.
- A future crosswalk between SEC identifiers and USAspending identifiers.

Key identifier issue:

- USAspending uses UEI and sometimes DUNS.
- SEC uses CIK and ticker.
- There is no automatic universal key between the two systems.

To combine SEC financial context with USAspending award exposure, the project will need a governed crosswalk table that maps company names, UEIs, DUNS values, CIKs, and tickers where appropriate.


## 13. Short Summary for Status Update

The USAspending prototype starts with a list of company names, finds matching parent entities, finds related child entities, and pulls award records tied to those entities. The refined script organizes the output into normalized files that support auditability and Power BI modeling.

Main governed output files:

- `entity_master.csv`: company/entity lookup table keyed by UEI.
- `relationships.csv`: child-to-parent relationship table.
- `award_fact.csv`: award-level fact table.

Main readable Power BI exports:

- `output/powerbi_prototype/entity_master.csv`
- `output/powerbi_prototype/award_fact_readable.csv`
- `output/powerbi_prototype/relationships_readable.csv`

The refined script also creates progress and failure logs so awards API issues can be diagnosed without assuming the entire run failed. If the awards step has a partial API failure, the normal recovery path is to rerun only the awards step and then refresh the Power BI exports.

The next technical direction is to compare the current API-by-UEI approach against a bulk download approach. Bulk downloads may be better for recurring dashboard refreshes because they reduce the number of live API calls and allow more processing to happen locally.

The SEC notebook remains a separate research path for financial context. Any combined USAspending/SEC analysis will require a maintained identifier crosswalk.


## 14. Completion Checklist

Use this checklist when handing the prototype to another analyst or when preparing a dashboard refresh.

Setup and input review:

- [ ] Confirm the notebook is opened from the repository root.
- [ ] Confirm `CompanyNames.xlsx` exists and contains the intended companies.
- [ ] Review the displayed company list in Step 2 before running API calls.

Pipeline execution:

- [ ] Run `parents` for the input companies.
- [ ] Review parent matches for obvious false positives.
- [ ] Run `children` to collect child/subsidiary entities.
- [ ] Run `combine` to create `company_hierarchy.csv`, `entity_master.csv`, and `relationships.csv`.
- [ ] Run `awards` to create or update `award_fact.csv`.
- [ ] If awards partially fail, review `failed_award_requests.csv` and rerun with `PIPELINE_RUN_MODE = "awards_only"`.
- [ ] Run `powerbi-exports` or confirm it ran as part of `--step all`.

Output validation:

- [ ] Review Step 4 file status for missing or unexpectedly small files.
- [ ] Review Step 5 metrics for entity counts, relationship counts, award rows, award amount, and failed requests.
- [ ] Confirm `output/powerbi_prototype` contains the readable export files.
- [ ] Confirm `run_log.csv` shows the expected completed steps.

Dashboard handoff:

- [ ] Use the normalized files for the governed Power BI model.
- [ ] Use readable exports when faster prototype labeling is needed.
- [ ] Keep progress and failure logs available for refresh troubleshooting.
- [ ] Compare the API-based award pull against the bulk download prototype before deciding on the production refresh strategy.
